In [13]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, END
from typing import TypedDict
from dotenv import load_dotenv
import os

In [11]:
def doc_loader(path):
    try:
        loader = DirectoryLoader(path,
                                glob="**/*.pdf",
                                loader_cls=PyMuPDFLoader,
                                show_progress=True)
        documents = loader.load()
        print(f"Loaded {len(documents)} documents from {path}")
        return documents
    except Exception as e:
        print(f"Error loading documents from {path}: {e}")
        return None

def text_splitter(documents):
    print("Splitting documents into chunks...")
    try:
        if not documents:
            raise ValueError("No documents to split")
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, 
                                              chunk_overlap=150,
                                              length_function=len,
                                              separators=["\n\n", "\n", " ", ""])
        chunks = splitter.split_documents(documents)
        print(f"""Split into {len(chunks)} chunks
            Document splitting complete""")
        return chunks
    except Exception as e:
        print(f"Error splitting documents: {e}")
        return None

def create_vector(chunks):
    print('Loading embedding model...')
    embedding = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
    print('Creating vector store...')
    vector_store = Chroma.from_documents(chunks, 
                                         embedding, 
                                         collection_name="pdf_docs")
    print('Vector store created successfully')
    return vector_store

def llm_model():
    print('Loading LLM model...')
    llm = ChatOpenAI(
        model="moonshotai/kimi-k2.6:free", 
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1")
    return llm

### using the above components to build Agentic RAG pipeline

In [18]:
# defining graph state
class agentstate(TypedDict):
    question:str
    rewritten_query:str
    documents:list
    web_result:str
    answer:str

### Re-writing the user query  

In [21]:
def rewrite(state):
    query = state["question"]
    llm = llm_model()
    prompt = f"""
    You are a query rewriting assistant.

    Rewrite the query only if it improves retrieval quality.

    Rules:
    - Preserve meaning
    - Do not add information
    - Return the original query if already clear

    Query:
    {query}
    """
    rewritten_query = llm.invoke(prompt)
    return {
        "rewritten_query":rewritten_query.content
    }


### Retrival Block

In [20]:
def retriever(state):
    documents = doc_loader("../data")
    chunks = text_splitter(documents)
    vector_db = create_vector(chunks)
    query = state["question"]
    retriever_db = vector_db.as_retriever()
    r_docs = retriever_db.invoke(query) 
    return {
        r_docs
    }